# The Travelling Salesman Problem (TSP) via Exhaustive Search

**Session:** 2026-10-28 · Brute-force Exhaustive  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

The **Travelling Salesman Problem (TSP)** is an NP-hard optimization problem: given a set of cities and pairwise distances, determine the shortest tour that visits every city exactly once and returns to the origin.

## Learning Outcomes

- Formulate TSP mathematically with adjacency distance matrices.
- Implement a brute-force exhaustive solver evaluating all $(n-1)!$ tours.
- Replicate and verify the 4-city lecture benchmark.
- Measure empirical wall-clock scaling on a 10-city dataset.


## 1. Mathematical Formulation

Let $V = \{0, 1, \dots, n - 1\}$ be the vertices (cities) and $D$ be an $n \times n$ distance matrix where $D_{ij}$ is the distance from city $i$ to city $j$.

Without loss of generality, fix the starting city at $s = 0$. Every tour is represented by a permutation $\pi = (c_1, c_2, \dots, c_{n-1})$ of the remaining $n - 1$ cities:
$$L(\pi) = D_{s, c_1} + \sum_{k=1}^{n-2} D_{c_k, c_{k+1}} + D_{c_{n-1}, s}$$

Exhaustive search evaluates all $(n - 1)!$ permutations to find:
$$\pi^* = \arg\min_\pi L(\pi)$$


In [ ]:
import json
from itertools import permutations
from pathlib import Path
from sys import maxsize


def solve_tsp_exhaustive(
    distances: list[list[float | int]], start_city: int = 0
) -> tuple[int | float, list[int], list[tuple[list[int], int | float]]]:
    """Exhaustive TSP solver returning optimal cost, tour, and all candidate evaluations."""
    num_cities = len(distances)
    other_cities = [i for i in range(num_cities) if i != start_city]

    min_path_weight = maxsize
    best_tour: list[int] = []
    all_evaluations: list[tuple[list[int], int | float]] = []

    for perm in permutations(other_cities):
        current_weight = 0
        k = start_city
        for next_city in perm:
            current_weight += distances[k][next_city]
            k = next_city
        current_weight += distances[k][start_city]

        full_tour = [start_city, *list(perm), start_city]
        all_evaluations.append((full_tour, current_weight))

        if current_weight < min_path_weight:
            min_path_weight = current_weight
            best_tour = full_tour

    return min_path_weight, best_tour, all_evaluations

## 2. Verification on Lecture Slide Benchmark (4 Cities)

The lecture slides define 4 cities with symmetric distances:
```text
    A   B   C   D
A   0   2   8   5
B   2   0   3   4
C   8   3   0   7
D   5   4   7   0
```


In [ ]:
# Load benchmark from data directory
data_file = Path("../data/tsp_distances_4.json")
with data_file.open("r", encoding="utf-8") as f:
    bench = json.load(f)

city_names = bench["cities"]
dist_matrix = bench["distance_matrix"]

min_weight, best_tour, all_tours = solve_tsp_exhaustive(dist_matrix, start_city=0)

print(f"Total candidate tours evaluated: {len(all_tours)}")
for tour, cost in all_tours:
    tour_named = " => ".join(city_names[i] for i in tour)
    print(f"  {tour_named} | Cost: {cost}")

best_tour_named = " => ".join(city_names[i] for i in best_tour)
print(f"\nOptimal Tour: {best_tour_named}")
print(f"Optimal Cost: {min_weight}")

assert min_weight == 17
assert min_weight == bench["optimal_cost"]
print("Lecture slide benchmark reproduced with 100% precision!")

## 3. Empirical Timing and Scalability Barrier

To demonstrate the factorial explosion, we benchmark exhaustive TSP for $n = 4, 6, 8, 9, 10$ using `tsp_cities_10.json`.


In [ ]:
import math
import time

data_10_file = Path("../data/tsp_cities_10.json")
with data_10_file.open("r", encoding="utf-8") as f:
    bench_10 = json.load(f)

dist_matrix_10 = bench_10["distance_matrix"]

print(f"{'Cities (n)':<12} {'Tours (n-1)!':<16} {'Time (ms)':<14} {'Best Cost':<10}")
print("-" * 55)

for n in [4, 6, 8, 9, 10]:
    sub_matrix = [row[:n] for row in dist_matrix_10[:n]]
    t0 = time.perf_counter()
    cost, _, _ = solve_tsp_exhaustive(sub_matrix, start_city=0)
    t1 = time.perf_counter()
    elapsed_ms = (t1 - t0) * 1000
    tours_count = math.factorial(n - 1)
    print(f"{n:<12} {tours_count:<16} {elapsed_ms:<14.2f} {cost:<10.2f}")